# Vector Databases — e-commerce product search, Pinecone + OpenAI embeddings

**Where we're headed.** By the end of this notebook, this line —

```python
index.query(vector=embed("wireless noise-cancelling headphones"), top_k=3)
```

— returns the **Sony WH-1000XM5**, **Bose QuietComfort 45**, and similar products from a 751-product catalog, even though none of their listings contain the phrase "noise-cancelling headphones" verbatim. That's the whole point of this notebook: the system finds *meaning*, not *keywords*.

Everything below builds up to that call, then goes past it into the operations a real system actually needs — updating, deleting, partitioning, and handling documents too long to embed in one piece.

Requires `OPENAI_API_KEY` and `PINECONE_API_KEY` set in your environment.

## 1 · The idea, in one line

> A **vector** is a list of numbers that carries the *meaning* of a sentence or word. We get that list from an **embedding model**. Similar meaning → similar numbers → nearby vectors — and finding nearby vectors is something a computer does very fast.

The score you'll see in results is a number from **−1 to +1**: closer to +1 means closer in meaning. You don't need to know how it's computed to use it correctly.

## 2 · The shape of a stored record

Every record Pinecone stores has three parts:

| Part | What it is | Example |
|---|---|---|
| `id` | a unique handle for the item | `"d2559c95-...-538c34a25bcb"` |
| `values` | the embedding — meaning, as coordinates | `[0.0123, -0.0456, ...]` (1536 numbers) |
| `metadata` | ordinary structured fields you filter on later | `{"price": 600.93, "currency": "USD"}` |

**Vectors find candidates. Metadata enforces rules.** Keep that split in mind — it's the pattern behind everything from here on.

## 3 · Set up the two clients

In [ ]:
# pip install openai pinecone pandas

import os
import pandas as pd
from openai import OpenAI      # turns text into vectors (the embedding model)
from pinecone import Pinecone, ServerlessSpec   # stores vectors and searches them

oa = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

EMBED_MODEL = "text-embedding-3-small"   # 1536 dims — must be the SAME model for every vector in this index
INDEX_NAME = "vidyasankalp-ecom-products"

def embed_batch(texts):
    """Turn a list of strings into a list of 1536-number vectors, one API call for the whole batch."""
    response = oa.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in response.data]   # same order as `texts`


## 4 · Load the dataset

**Dataset:** [kernel-memory-ecommerce-sample](https://github.com/yuniko-software/kernel-memory-ecommerce-sample) — 751 real product listings: `Name`, `Description`, `Price`, `PriceCurrency`, `SupplyAbility`, `MinimumOrder`.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/yuniko-software/kernel-memory-ecommerce-sample/main/utils/dataset/products.csv"

df = pd.read_csv(DATA_URL, sep="|")          # pipe-delimited, not comma
df["Description"] = df["Description"].fillna("")   # a couple of rows have no description

print(df.shape)
df.head()


## 5 · Decide what gets embedded, and what stays metadata

We embed `Name + Description` — that's where the meaning lives. `Price`, `PriceCurrency`, `SupplyAbility`, `MinimumOrder` become metadata: facts *about* the product, used for filtering, never for search.

In [ ]:
df["text"] = df["Name"] + ": " + df["Description"]
df[["Id", "text"]].head(3)


## 6 · Create the index (a one-time setup step)

In [ ]:
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,          # must match EMBED_MODEL's output size exactly
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-west-2"),
    )

index = pc.Index(INDEX_NAME)


## 7 · Embed and upsert all 751 products

Batch the embedding calls (one API call for many texts) and batch the upserts (Pinecone recommends ~100 records per call) — both standard practice at real scale.

In [ ]:
def to_metadata(row):
    return {
        "text": row["text"],
        "price": float(row["Price"]),
        "currency": row["PriceCurrency"],
        "supply_ability": int(row["SupplyAbility"]),
        "minimum_order": int(row["MinimumOrder"]),
    }

BATCH = 100
for start in range(0, len(df), BATCH):
    chunk = df.iloc[start:start + BATCH]
    vectors = embed_batch(chunk["text"].tolist())
    index.upsert(vectors=[
        {"id": row["Id"], "values": vec, "metadata": to_metadata(row)}
        for (_, row), vec in zip(chunk.iterrows(), vectors)
    ])

print(index.describe_index_stats())   # total vector count should read 751


## 8 · The payoff, for real this time

Note the query text shares almost no words with the winning product's listing — the match happens on meaning, not spelling.

In [ ]:
query_vector = embed_batch(["wireless noise-cancelling headphones"])[0]   # embed the QUERY with the SAME model as the data

result = index.query(vector=query_vector, top_k=5, include_metadata=True)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  {m["text"][:70]:70s}  {m["price"]:>8.2f} {m["currency"]}')


## 9 · Update and delete — keeping the index honest

`upsert` isn't just for adding new records. Every operation an index actually needs in production comes down to three facts:

- **Upsert with an existing `id` overwrites that record.** No error, no duplicate — this is how you re-index a product whose price or description changed. Same call as §7, just called again on an id that already exists.
- **`index.delete(ids=[...])`** removes specific records by id — e.g. a product that's been discontinued.
- **`index.delete(filter={...})`** removes every record matching a metadata condition — e.g. wiping an entire stale batch at once. This uses the exact same filter syntax as querying (§10), which is worth noticing: filters mean the same thing whether you're searching or deleting.

We'll demonstrate on two throwaway records, tagged `"demo": True` in their metadata, so nothing below touches the real 751-product catalog.

In [ ]:
# --- create two small demo records, clearly marked so they never get confused with real data ---
demo_vectors = embed_batch([
    "Demo Product A: a placeholder item used only to demonstrate update and delete.",
    "Demo Product B: a placeholder item used only to demonstrate update and delete.",
])

index.upsert(vectors=[
    {"id": "demo-1", "values": demo_vectors[0], "metadata": {"text": "Demo Product A", "price": 10.0, "demo": True}},
    {"id": "demo-2", "values": demo_vectors[1], "metadata": {"text": "Demo Product B", "price": 20.0, "demo": True}},
])

print(index.fetch(ids=["demo-1"])["vectors"]["demo-1"]["metadata"])   # {'text': 'Demo Product A', 'price': 10.0, 'demo': True}


In [ ]:
# --- UPDATE: upserting the same id again overwrites it in place, no duplicate is created ---
updated_vector = embed_batch(["Demo Product A: now on clearance, price reduced."])[0]

index.upsert(vectors=[
    {"id": "demo-1", "values": updated_vector, "metadata": {"text": "Demo Product A (clearance)", "price": 4.99, "demo": True}},
])

print(index.fetch(ids=["demo-1"])["vectors"]["demo-1"]["metadata"])   # price is now 4.99 — same id, updated content


In [ ]:
# --- DELETE by id: remove one specific record ---
index.delete(ids=["demo-2"])

try:
    index.fetch(ids=["demo-2"])["vectors"]["demo-2"]
    print("still present (unexpected)")
except KeyError:
    print("demo-2 is gone")


In [ ]:
# --- DELETE by filter: remove everything matching a metadata condition, in one call ---
# scoped to demo:True so this can never accidentally touch the real catalog
index.delete(filter={"demo": {"$eq": True}})

print(index.describe_index_stats())   # total count is back to 751 — both demo records are gone


## 10 · Metadata filtering — worth its own section

Real search is rarely just "closest vector." It's "closest vector that also costs under $500 and is well-stocked." This is the same **vectors find candidates, metadata enforces rules** split from §2 — and it's a pattern you'll reuse constantly: access control (filter by tenant), freshness (filter by date), inventory (filter by stock), permissions (filter by role).

**The operators available:**

| Operator | Meaning | Example |
|---|---|---|
| `$eq` | equals | `{"currency": {"$eq": "USD"}}` |
| `$ne` | not equal | `{"currency": {"$ne": "EUR"}}` |
| `$gt` / `$gte` | greater than / or equal | `{"price": {"$gte": 100}}` |
| `$lt` / `$lte` | less than / or equal | `{"price": {"$lt": 500}}` |
| `$in` | value is one of a list | `{"currency": {"$in": ["USD", "EUR"]}}` |
| `$nin` | value is none of a list | `{"currency": {"$nin": ["JPY"]}}` |

**Combining conditions:** multiple keys in one `filter={}` dict are **AND**ed together automatically. For **OR** logic, wrap conditions explicitly with `$or`.

Two pitfalls: **type must match exactly** (filtering `price` with a string instead of a float silently matches nothing, not an error), and **you can only filter on fields stored as metadata at upsert time** — there's no adding a filterable field after the fact without re-upserting.

In [ ]:
# Example 1 — AND: currency is USD, price is under 500, and supply_ability is at least 1000
result = index.query(
    vector=embed_batch(["a laptop for creative professional work"])[0],
    top_k=5,
    include_metadata=True,
    filter={
        "currency": {"$eq": "USD"},
        "price": {"$lt": 500},
        "supply_ability": {"$gte": 1000},
    },
)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  {m["text"][:70]:70s}  {m["price"]:>8.2f} {m["currency"]}')


**Example 2 — `$in`:** products priced in either currency, but only the cheap end of the catalog.

In [ ]:
result = index.query(
    vector=embed_batch(["a sturdy gift for someone who works outdoors"])[0],
    top_k=5,
    include_metadata=True,
    filter={
        "currency": {"$in": ["USD", "EUR"]},
        "price": {"$lt": 100},
    },
)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  {m["text"][:70]:70s}  {m["price"]:>8.2f} {m["currency"]}')


**Example 3 — explicit `$or`:** either a cheap USD item, or any item with very high supply (bulk-order friendly), regardless of price.

In [ ]:
result = index.query(
    vector=embed_batch(["a reliable everyday backpack"])[0],
    top_k=5,
    include_metadata=True,
    filter={
        "$or": [
            {"currency": {"$eq": "USD"}, "price": {"$lt": 100}},
            {"supply_ability": {"$gte": 5000}},
        ]
    },
)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  {m["text"][:70]:70s}  {m["price"]:>8.2f} {m["currency"]}  (supply {m["supply_ability"]})')


## 11 · Namespaces — partitioning one index

A **namespace** splits one index into isolated sub-spaces. A query only ever searches within the namespace you name — records in other namespaces are invisible to it, with zero extra infrastructure. The standard uses:

- **Multi-tenant systems** — each customer's data in its own namespace, so tenant A can never see tenant B's vectors, even by accident.
- **Environment separation** — `dev` and `prod` namespaces in the same index, instead of paying for two indexes.
- **Separate datasets sharing one index** — e.g. a product catalog namespace and a support-docs namespace, if both happen to use the same embedding model.

This solves the same problem your ACT platform solves with a single-table DynamoDB design and typed sort keys — different mechanism (physical partitioning vs. key structure), same goal: keep tenants' data cleanly separated without multiplying infrastructure.

We'll demonstrate by splitting a small slice of the catalog into two namespaces by currency — a stand-in for "US storefront" vs. "EU storefront".

In [ ]:
usd_sample = df[df["PriceCurrency"] == "USD"].head(10)
eur_sample = df[df["PriceCurrency"] == "EUR"].head(10)

for sample, ns in [(usd_sample, "storefront-us"), (eur_sample, "storefront-eu")]:
    vectors = embed_batch(sample["text"].tolist())
    index.upsert(
        vectors=[
            {"id": row["Id"], "values": vec, "metadata": to_metadata(row)}
            for (_, row), vec in zip(sample.iterrows(), vectors)
        ],
        namespace=ns,   # <-- the only difference from a normal upsert
    )

print(index.describe_index_stats())   # namespaces breakdown shows storefront-us: 10, storefront-eu: 10, '' (default): 751


In [ ]:
# Querying is scoped to one namespace at a time — results from other namespaces never appear
result = index.query(
    vector=embed_batch(["a comfortable running shoe"])[0],
    top_k=3,
    include_metadata=True,
    namespace="storefront-us",   # <-- only searches the 10 records upserted above under this namespace
)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  {m["text"][:70]:70s}  {m["price"]:>8.2f} {m["currency"]}')

# Omitting `namespace=` (as in every query earlier in this notebook) searches the default namespace, "" —
# which is where all 751 products from §7 live. storefront-us and storefront-eu are additive, not a split.


## 12 · Chunking long text before embedding

Every embedding model has a maximum input length. `text-embedding-3-small` accepts up to 8,191 tokens (~6,000 words) — generous, but real documents blow past it: a 40-page PDF, a legal filing, a full product manual. Text beyond the limit isn't rejected — it's **silently truncated**, and the tail of the document never influences the vector at all. This is one of the most common real-world bugs in retrieval systems, and it produces no error message.

Our product descriptions were short enough to dodge this entirely. Here's the fix for when they aren't: split long text into overlapping chunks *before* embedding, embed each chunk separately, and store each as its own vector — with metadata linking it back to the source document.

In [ ]:
def chunk_text(text, chunk_size=60, overlap=15):
    """Split text into overlapping word-count chunks.
    chunk_size=60, overlap=15 are small on purpose, to make the effect visible in this demo —
    real settings are usually a few hundred words with 10-20% overlap.
    The overlap matters: without it, a sentence split across a chunk boundary can lose its meaning
    in both halves. With it, the sentence survives intact in at least one chunk.
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap   # step forward by less than chunk_size, so chunks overlap
    return chunks

# a synthetic "long document" — several product descriptions concatenated, standing in for
# something like a 40-page manual that would otherwise exceed the embedding model's input limit
long_document = " ".join(df["Description"].head(15).tolist())
print(f"{len(long_document.split())} words total")

chunks = chunk_text(long_document)
print(f"split into {len(chunks)} chunks")
print(chunks[0][:200], "...")


In [ ]:
# Embed and upsert each chunk as its own vector — metadata links every chunk back to its source
# document and records its position, so a matching chunk can be traced back to where it came from.
chunk_vectors = embed_batch(chunks)

index.upsert(vectors=[
    {
        "id": f"demo-doc-1-chunk-{i}",
        "values": vec,
        "metadata": {"text": chunk, "source_doc": "demo-doc-1", "chunk_index": i, "demo": True},
    }
    for i, (chunk, vec) in enumerate(zip(chunks, chunk_vectors))
])

# query against the chunks specifically, using the same demo:True tag so this stays isolated
result = index.query(
    vector=embed_batch(["information about battery life and charging"])[0],
    top_k=2,
    include_metadata=True,
    filter={"source_doc": {"$eq": "demo-doc-1"}},
)

for match in result["matches"]:
    m = match["metadata"]
    print(f'{match["score"]:.3f}  chunk {m["chunk_index"]}: {m["text"][:100]}...')

# clean up the demo chunks — same delete-by-filter pattern from §9
index.delete(filter={"demo": {"$eq": True}})


## 13 · Why this works, briefly

- **The score is cosine similarity.** −1 to +1; +1 means "same meaning."
- **Query and data must share one embedding model.** `EMBED_MODEL` was used everywhere in this notebook — swap models and "nearest" silently stops meaning anything.
- **This stays fast even at millions of records.** Pinecone uses an approximate index (HNSW), trading a small, tunable amount of accuracy for a large speed-up.

Covered in full in the companion book: cosine similarity in Part II, the one-model rule in Part V §24, the scale problem in Part VI.

## Recap

- `text` (name + description) is what gets embedded; everything else is metadata.
- `upsert` both creates and updates — same id in, overwritten in place. `delete(ids=[...])` and `delete(filter={...})` remove records, by id or in bulk.
- Metadata filtering (`$eq`, `$in`, `$or`, ...) is a reusable pattern: vectors find candidates, metadata enforces rules.
- Namespaces partition one index for tenants, environments, or datasets — a query only ever sees its own namespace.
- Long documents must be chunked with overlap before embedding, or the tail is silently dropped.

**Want the exact math** behind cosine similarity, the dot product, and BM25? That's Appendix A of the companion book — optional.